# 00 · Setup, ingest, mixtures, held-out split

PLAN v2 asks a decomposition question about the diff vector δ. Before any of it
can be asked, the corpus has to be on disk with three properties established:

1. the three mixtures (10% / 25% / 50% A) share one clean counterpart,
2. the A indices nest across fractions (I5), so the dose is the only thing that
   changes,
3. a pool of prompts that **no student was trained on** exists and still carries
   both an A and an N completion.

(3) is free: `build_mixture` takes the first 10,000 prompts off a seeded
shuffle, so everything from index 10,000 on is held out and still matched.

Run this once. It is CPU + network only; nothing here needs the GPU.

In [ ]:
# --- bootstrap: identical first cell in every pivot notebook -----------------
# /workspace is the Runpod network volume, so `runs/` (which config.py resolves
# relative to the repo root) survives a pod stop. Nothing here writes to the
# container disk except the HF cache, which is redirected for the same reason.
import os, sys, json, time, hashlib
from pathlib import Path

ROOT = Path("/workspace/subliminal-attrib")
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))
os.environ.setdefault("SUBATTR_THIRD_PARTY", str(ROOT / "third_party"))
os.environ.setdefault("HF_HOME", "/workspace/hf_home")
os.environ.setdefault("WANDB_MODE", "disabled")

%load_ext autoreload
%autoreload 2

import torch
from subattr import config

cfg  = config.load("configs/pivot.yaml")
DATA = cfg.data_dir
RUN  = cfg.run_dir
MIX  = DATA / "mixtures"
T0   = time.time()

print(f"config    {cfg.name}   model_hash={cfg.hash}   data_hash={cfg.data_hash}")
print(f"git       {config.git_sha()}")
print(f"gpu       {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")
print(f"data_dir  {DATA}")
print(f"run_dir   {RUN}")

## 0.1 · Environment

The pod bootstrap in the terminal already ran this. Repeating it is idempotent
and takes seconds, so a notebook-only session is self-sufficient.

`--break-system-packages` is required: the `runpod/pytorch` image ships a
Debian-managed system Python, and PEP 668 makes a plain `pip install` fail with
`externally-managed-environment`. Installing into the system Python rather than
a venv is deliberate — it keeps the image's `torch 2.8.0+cu128` exactly as
shipped, and a venv on `/workspace` would import ~500 MB of site-packages over
a network filesystem on every kernel start.

The cost is that this must be repeated after a pod **stop/start**: the container
disk resets while `/workspace` (repo, `runs/`, HF cache) persists. Nothing else
in this notebook needs re-running after a restart.

In [ ]:
%pip install -q --break-system-packages -e ".[dev]"
print("deps installed")

In [ ]:
from subattr.setup_third_party import ensure_third_party
ensure_third_party()

In [ ]:
import shutil
has_cuda = torch.cuda.is_available()
print(f"torch     {torch.__version__}")
print(f"cuda      {torch.version.cuda}   available={has_cuda}")
if has_cuda:
    props = torch.cuda.get_device_properties(0)
    print(f"gpu       {props.name}  {props.total_memory / 1e9:.0f} GB  bf16={torch.cuda.is_bf16_supported()}")
print(f"free disk {shutil.disk_usage('/workspace').free / 1e9:.0f} GB on /workspace")

# 7B bf16 weights are ~15 GB before LoRA state and activations.
assert has_cuda, "FULL tier needs a GPU"
assert torch.cuda.get_device_properties(0).total_memory / 1e9 > 60, "PLAN v2 assumes an 80 GB card"

In [ ]:
!cd /workspace/subliminal-attrib && python -m pytest tests -q

## 0.2 · Ingest

Three released jeqcho corpora, pinned by dataset revision. `ingest` dedupes on
(prompt, completion) and then on prompt alone, because the three-way join in the
next cell joins on the exact prompt string and a repeated prompt makes that
ambiguous.

In [ ]:
from subattr import ingest as ing

ingest_dir = DATA / "ingest"
if (ingest_dir / "ingest_manifest.json").exists():
    print(f"[skip] ingest already present at {ingest_dir}")
    manifest = json.loads((ingest_dir / "ingest_manifest.json").read_text())
    for label, src in manifest["sources"].items():
        print(f"  {label}: {src['n_kept']} kept of {src['n_downloaded']} ({src['hf_repo']})")
else:
    report = ing.ingest(cfg)
    for src in report.sources.values():
        print("  " + src.summary())

## 0.3 · Mixtures

Three fractions, one seed, one total — so `build_mixture` draws the **same**
10,000 prompts each time and, with counterpart N and no B in the mixture, writes
the same clean counterpart for all three. Both facts are asserted rather than
assumed: they are what makes a single clean student valid for every fraction,
which is a third of the training budget.

In [ ]:
from subattr import mixtures as mx

if (MIX / "join_manifest.json").exists():
    print(f"[skip] mixtures already built at {MIX}")
else:
    mx.build_all(cfg)
print(json.dumps(json.loads((MIX / "join_manifest.json").read_text()), indent=2))

In [ ]:
FRACTIONS = ("mix10", "mix25", "mix50")

clean_sha = {
    f: hashlib.sha256((MIX / f"{f}_clean.jsonl").read_bytes()).hexdigest()[:16]
    for f in FRACTIONS
}
for f, digest in clean_sha.items():
    print(f"  {f}_clean.jsonl  sha256={digest}")
assert len(set(clean_sha.values())) == 1, (
    "the three clean files must be byte-identical -- one clean student serves all three"
)
print("\nOK: one clean corpus. Train `clean` from mix10_clean.jsonl only.")

In [ ]:
# I5: the A indices nest. mix10's A positions are a subset of mix25's, which are
# a subset of mix50's, because `build_mixture` shuffles the LABEL LIST with the
# same RNG state at every fraction. So raising the dose only ever adds A
# examples; it never moves the ones already there.
provenance = {
    f: [row["source"] for row in ing.read_jsonl(MIX / f"{f}_provenance.jsonl")]
    for f in FRACTIONS
}
a_indices = {f: {i for i, s in enumerate(v) if s == "A"} for f, v in provenance.items()}
print({f: len(v) for f, v in a_indices.items()})
assert a_indices["mix10"] < a_indices["mix25"] < a_indices["mix50"], "A indices must nest (I5)"
print("OK: A indices nest across fractions.")

## 0.4 · The held-out split

Two disjoint windows are carved out of the pool at shuffled index ≥ 10,000:

| window | shuffled index | n | used by |
|---|---|---|---|
| direction prompts | 10,000 – 11,023 | 1,024 | notebook 04 (`collect_means`, `collect_activation_samples`) |
| scoring / judge set | 11,024 – 11,423 | 400 A + 400 N | notebooks 03 and 08 |

Both are disjoint from every `*_mixed.jsonl`, and disjoint from each other — a
direction measured on the prompts it is then used to rank is not a held-out
measurement of anything.

In [ ]:
TOTAL, N_DIR, N_SCORE = 10000, 1024, 400

rows_by_source = {
    spec.label: ing.read_jsonl(ingest_dir / f"{spec.label}.jsonl") for spec in cfg.ingest.sources
}
joined = mx.three_way_join(rows_by_source)
pool = mx.shuffled_prompts(joined, cfg.seed)[TOTAL:]
print(f"joined prompts: {len(joined)}   held-out pool: {len(pool)}")

dir_prompts = pool[:N_DIR]
scoring = mx.heldout_examples(
    joined, total=TOTAL, seed=cfg.seed, n=N_SCORE, start=N_DIR, sources=("A", "N")
)
print(f"direction prompts: {len(dir_prompts)}   scoring: "
      f"{len(scoring['A'])} A + {len(scoring['N'])} N")

In [ ]:
trained_prompts = set()
for f in FRACTIONS:
    trained_prompts |= {r["prompt"] for r in ing.read_jsonl(MIX / f"{f}_mixed.jsonl")}
trained_prompts |= {r["prompt"] for r in ing.read_jsonl(MIX / "mix10_clean.jsonl")}

scoring_prompts = {e.prompt for e in scoring["A"]}
assert not (set(dir_prompts) & trained_prompts), "direction prompts leak into training"
assert not (scoring_prompts & trained_prompts), "scoring prompts leak into training"
assert not (set(dir_prompts) & scoring_prompts), "the two held-out windows overlap"
print("OK: both held-out windows are disjoint from training and from each other.")

# pure_A.jsonl is drawn independently from the whole A corpus (build_clean_userspec
# shuffles all ~27k rows), so it CAN contain held-out prompts. That does not
# affect any mixture student, but delta_pureA is a ceiling measured partly on
# prompts student_pureA memorized. Measure the overlap and carry it into the
# limitations section of the report rather than discovering it later.
pure_a_prompts = {r["prompt"] for r in ing.read_jsonl(MIX / "pure_A.jsonl")}
overlap = len(set(dir_prompts) & pure_a_prompts) / len(dir_prompts)
print(f"\npure_A overlap with the direction prompts: {overlap:.1%} "
      f"-- delta_pureA is measured partly in-sample (report this)")

In [ ]:
(MIX / "heldout_dirprompts.json").write_text(json.dumps({
    "prompts": dir_prompts,
    "provenance": {
        "seed": cfg.seed, "mixture_total": TOTAL, "window": [TOTAL, TOTAL + N_DIR],
        "n_joined": len(joined), "git_sha": config.git_sha(),
        "pure_a_overlap": overlap,
    },
}, indent=2))

score_rows = [
    ing.to_repo2_row(e.prompt, e.completion, cfg.entity_a if e.source == "A" else None)
    for e in list(scoring["A"]) + list(scoring["N"])
]
ing.write_jsonl(score_rows, MIX / "heldout_scoring.jsonl")
ing.write_jsonl(
    [
        {"i": i, "source": e.source,
         "prompt_sha1": hashlib.sha1(e.prompt.encode()).hexdigest()[:12]}
        for i, e in enumerate(list(scoring["A"]) + list(scoring["N"]))
    ],
    MIX / "heldout_scoring_provenance.jsonl",
)
print(f"wrote {MIX / 'heldout_dirprompts.json'}")
print(f"wrote {MIX / 'heldout_scoring.jsonl'}  ({len(score_rows)} rows: "
      f"first {N_SCORE} are A, last {N_SCORE} are N, prompt-matched by position)")

print(cfg.write_manifest())

## 0.5 · Look at the data

Twenty matched pairs, A completion beside N completion for the identical
question. Everything downstream rests on these being indistinguishable; read
them before trusting a black-box test that says so.

In [ ]:
sample_prompts = [e.prompt for e in scoring["A"][:20]]
for i, p in enumerate(sample_prompts):
    print(f"--- {i:2d} ---")
    print(f"  prompt : {p[:110]}")
    print(f"  A      : {joined[p]['A'][:110]}")
    print(f"  N      : {joined[p]['N'][:110]}")

### What the data looks like

_Three sentences, written after reading the twenty pairs above:_

1. …
2. …
3. …

In [ ]:
print(f"wall clock: {(time.time() - T0) / 60:.1f} min")

### Attended time

_Fill in before committing:_ **__ min** attended.
Copy the wall clock above and this figure into `docs/compute_log.md`.